In [311]:
def read_vrp(file):
    coords = {}
    demand = {}
    capacity = None
    depot = None

    with open(file, "r") as f:
        section = None

        for line in f:
            line = line.strip()
            if "CAPACITY" in line:
                capacity =int(line.split(":")[1])
            elif line == "NODE_COORD_SECTION":
                section = "coords"
                continue 
            elif line == "DEMAND_SECTION":
                section = "demand"
                continue 
            elif line == "DEPOT_SECTION":
                section = "depot"
                continue 
            elif line == "EOF":
                break

            if section == "coords":
                parts = line.split()
                node = int(parts[0])
                x = float(parts[1])
                y = float(parts[2])
                coords[node] = (x,y)
            elif section == "demand":
                parts = line.split()
                node = int(parts[0])
                demand[node] = int(parts[1])
            elif section == "depot":
                if line != "-1":
                    depot = int(line)
                
    return coords, demand, capacity, depot

In [312]:
import math
def Distance_matrix(coords):
    dist = {}
    for i in coords:
          for j in coords:
            x1, y1 = coords[i]
            x2, y2 = coords[j]
            dist[(i, j)] = sqrt((x1 - x2)**2 + (y1 - y2)**2)
    return dist

In [313]:
def cost(solution, dist, depot):
    cost = 0
    for route in solution:
        if not route:
            return 0
            
        cost += dist[(depot, route[0])]

        for i in range(len(route)-1):
            cost+= dist[(route[i], route[i+1])]
        
        cost += dist[(route[-1], depot)]
    
    return cost
    
def total_cost(solution, dist, depot):
    return sum(cost(solution, dist, depot) for route in solution)

In [314]:
def feasible(solution, demand, capacity):
    for route in solution:
        load = sum(demand[node] for node in route) 
        if load > capacity:
            return False
    return True

In [315]:
import random
def initial(coords, demand, capacity, depot):
    nodes = [i for i in coords if i != depot]
    random.shuffle(nodes)
    solution = []
    current_route = []
    load = 0

    while nodes:
        for i in nodes:
            if load + demand[i] <= capacity:
                current_route.append(i)
                load += demand[i]
                nodes.remove(i)
        else:
            solution.append(current_route)
            current_route = [i]

        if current_route:
            solution.append(current_route)
            current_route = []
            load = 0
            
    return solution
        

In [344]:
def initial(coords, demand, capacity, depot):
    nodes = [i for i in demand if i != depot] 
    random.shuffle(nodes)

    solution = []
    current_route = []
    load = 0

    for node in nodes:
        dem = demand[node]
        if load + dem <= capacity:
            current_route.append(node)
            load += dem
        else:
            if current_route:
                solution.append(current_route)
            current_route = [node]
            load = dem

    if current_route:
        solution.append(current_route)


    return solution

In [386]:
def bestof(initial, dist, demand, capacity, depot, mode="N1", runs=50):
    best_overall = None
    best_cost_overall = float("inf")
    
    for _ in range(runs):
        solution, cost_val = anneal(initialsol, dist, demand, capacity, depot, mode=mode)
        
        if cost_val < best_cost_overall:
            best_cost_overall = cost_val
            best_overall = solution
    
    return best_overall, best_cost_overall

In [9]:
import os
print(os.path.exists("C:/Users/aidan_iel1yrj/Downloads/cvrp-instances (1)/eil22.vrp"))

False


In [11]:
print(os.getcwd())

/home/aidan_iel1yrj


In [318]:
vrp_data = {
    "eil22": read_vrp("eil22.vrp"),
    "eil23": read_vrp("eil23.vrp"),
    "eil30": read_vrp("eil30.vrp"),
    "eil33": read_vrp("eil33.vrp")
}

D = {
    "eil22": Distance_matrix(vrp_data["eil23"][0]),
    "eil23": Distance_matrix(vrp_data["eil23"][0]),
    "eil30": Distance_matrix(vrp_data["eil23"][0]),
    "eil33": Distance_matrix(vrp_data["eil23"][0])
}

In [319]:
print(vrp_data["eil22"])
print(D["eil22"])

({1: (145.0, 215.0), 2: (151.0, 264.0), 3: (159.0, 261.0), 4: (130.0, 254.0), 5: (128.0, 252.0), 6: (163.0, 247.0), 7: (146.0, 246.0), 8: (161.0, 242.0), 9: (142.0, 239.0), 10: (163.0, 236.0), 11: (148.0, 232.0), 12: (128.0, 231.0), 13: (156.0, 217.0), 14: (129.0, 214.0), 15: (146.0, 208.0), 16: (164.0, 208.0), 17: (141.0, 206.0), 18: (147.0, 193.0), 19: (164.0, 193.0), 20: (129.0, 189.0), 21: (155.0, 185.0), 22: (139.0, 182.0)}, {1: 0, 2: 1100, 3: 700, 4: 800, 5: 1400, 6: 2100, 7: 400, 8: 800, 9: 100, 10: 500, 11: 600, 12: 1200, 13: 1300, 14: 1300, 15: 300, 16: 900, 17: 2100, 18: 1000, 19: 900, 20: 2500, 21: 1800, 22: 700}, 6000, 1)
{(1, 1): 0.0, (1, 2): 47.01063709417264, (1, 3): 41.88078318274385, (1, 4): 49.73932046178355, (1, 5): 62.625873247404705, (1, 6): 64.4437739428721, (1, 7): 35.77708763999664, (1, 8): 27.784887978899608, (1, 9): 45.0, (1, 10): 37.12142238654117, (1, 11): 33.52610922848042, (1, 12): 22.02271554554524, (1, 13): 7.0710678118654755, (1, 14): 30.805843601498726

In [320]:
#neighbourhood operator N1
def N1(solution):
    newsolution = [route[:] for route in solution]
    route =random.choice(newsolution)

    if len(route) <= 3:
        return newsolution
    i,j = random.sample(range(len(route)),2)
    route[i], route[j] = route[j], route[i]

    return newsolution

In [368]:
import random
def N2(solution, demand, capacity):
    newsolution = [route[:] for route in solution]
    if len(newsolution)<2:
        return newsolution
        
    r1, r2 =random.sample(newsolution, 2)

    if len(r1) <= 1:
        return newsolution
    i = random.randrange(len(r1))
    node = r1.pop(i)

    j = random.randrange(len(r2)+1)
    r2.insert(j, node)

    #feasibility check
    if feasible(newsolution, demand, capacity):
        return newsolution
    else:
        return solution

In [322]:
def anneal(initial, dist, demand, capacity, depot, T0=5000, alpha=0.9995, max_iter=5000, mode="N1"):
    current = [route[:] for route in initial]
    best =  [route[:] for route in initial]
    currentcost = cost(current, dist, depot)
    bestcost = currentcost
    T = T0

    for _ in range(max_iter):
        if mode == "N1":
            candidate = N1(current)
        elif mode == "N2":
            candidate = N2(current, demand, capacity)
        elif mode == "N1N2":
            if random.random() < 0.5:
                candidate = N1(current)
            else:
                candidate = N2(current, demand, capacity)
        else:
            candidatecost = solution_cost(candidate, D)
        if not feasible(candidate, demand, capacity):
            candidate = current
            
        candidatecost = cost(candidate, dist, depot)

        delta = total_cost(candidate, dist, depot) - total_cost(current, dist, depot)
            
        if delta < 0 or random.random() < math.exp(-delta / T):
            current = candidate
            currentcost = candidatecost
        if currentcost < bestcost:
            best = [route[:] for route in current]
            bestcost = currentcost
        T *= alpha
    return best, bestcost

In [345]:
vrp_data = {
    "eil22": read_vrp("eil22.vrp"),
    "eil23": read_vrp("eil23.vrp"),
    "eil30": read_vrp("eil30.vrp"),
    "eil33": read_vrp("eil33.vrp")
}

D = {
    "eil22": Distance_matrix(vrp_data["eil22"][0]),
    "eil23": Distance_matrix(vrp_data["eil23"][0]),
    "eil30": Distance_matrix(vrp_data["eil30"][0]),
    "eil33": Distance_matrix(vrp_data["eil33"][0])
}

coords, demand, capacity, depot = vrp_data["eil22"]
dist22 = D["eil22"]
initialsol = initial(coords, demand, capacity, depot)
print(initialsol)

[[15, 13, 2, 20], [5, 17, 3], [6, 21, 4, 7], [12, 9, 16, 8, 14, 11, 18], [10, 19, 22]]


In [324]:
print(initialsol)

[[19, 8, 7, 2, 15, 4], [17, 5, 16, 11], [14, 6, 10, 13, 3, 9], [20, 21, 12], [18, 22]]


In [390]:
print("N1",anneal(initialsol, dist22, demand, capacity, depot, mode="N1"))
print("N2",anneal(initialsol, dist22, demand, capacity, depot, mode="N2"))
print("N1N2",anneal(initialsol, dist22, demand, capacity, depot, mode="N1N2"))

N1 ([[13, 2, 15, 20], [5, 17, 3], [21, 6, 4, 7], [16, 18, 14, 12, 8, 9, 11], [10, 19, 22]], 841.5768020053622)
N2 ([[9, 7, 4, 6, 3], [22, 18, 16, 14], [20, 11, 5, 2], [8, 10, 13, 17, 15], [21, 19, 12]], 657.37260315146)
N1N2 ([[10, 13, 17, 19, 16], [15, 14, 3, 7], [21, 18, 20, 22], [12, 5, 8, 11], [6, 4, 2, 9]], 623.9794765708014)


In [387]:
print("initial cost", cost(initialsol))
print("N1",bestof(initialsol, dist22, demand, capacity, depot, mode="N1"))
print("N2",bestof(initialsol, dist22, demand, capacity, depot, mode="N2"))
print("N1N2",bestof(initialsol, dist22, demand, capacity, depot, mode="N1N2"))

N1 ([[2, 13, 20, 15], [5, 17, 3], [21, 6, 7, 4], [18, 16, 8, 11, 9, 12, 14], [10, 19, 22]], 818.6331032788241)
N2 ([[19, 21, 16], [20, 22, 18, 12], [5, 4, 9], [11, 8, 2, 3, 6, 7], [13, 17, 14, 15, 10]], 546.7938714434132)
N1N2 ([[13, 6, 2], [16, 11, 9], [18, 17, 14, 15], [19, 21, 20, 22], [7, 5, 4, 12, 3, 8, 10]], 555.8860009633664)


In [393]:
print("initial cost", cost(initialsol, dist22, depot))

initial cost 965.4117641852935
